# Fairness-Aware GPT-2 — QQP Training

Trains the paraphrase models from the report and produces the Table 2 numbers.
Scope is **QQP only** — that's where the fairness contribution is.

**Before anything: Runtime → Change runtime type → T4 GPU.**

Run cells **in order, top to bottom**. Each depends on the last.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Set up

**Edit `REPO` below, then run each cell.**

Colab clones from GitHub — anything you haven't pushed won't be here.

In [ ]:
REPO = "https://github.com/Ricky11234/fairness-aware-gpt2"

In [ ]:
!pip install -q uv

### Clean clone

`rm -rf` first. A leftover clone from an earlier session makes `git clone` fail
with *"destination path already exists"* and silently leaves you on stale code —
which matters, because `identity.py` generates the counterfactuals *and* computes
the flip rate.

In [ ]:
%cd /content
!rm -rf fairness-repo
!git clone -q $REPO fairness-repo

In [ ]:
%cd fairness-repo

### Verify you got current code

Must print a number **1 or greater**. If it prints `0`, you haven't pushed —
go to GitHub Desktop and hit **Push origin**, then re-run the clone cell.

In [ ]:
!grep -c AMBIGUOUS_PRONOUNS src/fairness_gpt2/identity.py

In [ ]:
!uv sync --group train

### Install CUDA torch

`pyproject.toml` pins torch to PyTorch's **CPU** index — deliberately, so the
Streamlit Cloud build stays under its 1GB limit. Colab needs the CUDA build.

`--reinstall` is required. Without it `uv pip install torch` sees torch already
present, decides the requirement is satisfied, and does nothing.

**This should take several minutes** (~2.5GB of CUDA wheels). If it finishes in
0s, it no-opped again — stop.

In [ ]:
!uv pip install --reinstall torch --torch-backend=auto

### Freeze the environment

`uv run` re-syncs from `uv.lock` before every command — which would throw away
the CUDA torch you just installed and put the CPU wheel back. This stops that.

It goes **after** the reinstall, not before.

In [ ]:
%env UV_NO_SYNC=1

### Verify — this is the gate

In [ ]:
!uv run --no-sync python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())" 

**Must print `cuda: True`.** If it still says `False`, try pinning the backend
explicitly instead of `auto` (uncomment below), then re-run the verify cell.

In [ ]:
# Fallback only if --torch-backend=auto misdetected the driver:
# !uv pip install --reinstall torch --torch-backend=cu126

## 2. Download QQP

`--train-size 283011` matches the report's pair count (GLUE ships 363,846) and
cuts ~29% off every run. Dev stays at 40,430 — exactly the report's.

In [ ]:
!uv run scripts/download_data.py --only qqp --train-size 283011 --skip-test

## 3. Smoke test — do not skip

Two minutes. Catches setup problems before you spend two hours finding them.

In [ ]:
!uv run fairness-train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out /tmp/smoke --epochs 1 --train-subset 2000 --eval-subset 1000

### Check three things

1. **`device=cuda`** — the one that matters. If it says `cpu`, stop.
2. **`dev_acc` ≈ 0.6–0.7** — correct. It's 1 epoch on 2,000 of 283,011 pairs.
   You're checking that it *runs*, not that it's good.
3. **`fairness_loss` > 0** — if exactly `0.0`, no counterfactuals were generated
   and `cda_reg` has quietly degraded into plain CDA.

## 4. Train `cda_reg` — the model your app deploys

~2 hours on a T4. `--save-half` writes fp16 (~250MB) for Streamlit Cloud's
memory limit.

**Keep this tab open and visible.** Free Colab kills idle sessions.

In [ ]:
!uv run fairness-train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/cda_reg --epochs 5 --lambda-fair 0.5 --save-half

## 5. Save results now

Writes `results/reproduced/cda_reg.json` — that's what makes your dashboard show
**your** numbers beside the paper's. Download before starting anything else, in
case the session dies.

In [ ]:
!zip -qr reproduced.zip results/reproduced
from google.colab import files
files.download("reproduced.zip")

## 6. Push the model to Hugging Face

Needs a **Write** token from huggingface.co/settings/tokens.

In [ ]:
!uv run huggingface-cli login

In [ ]:
HF_USER = "YOUR-HF-USERNAME"   # <-- EDIT THIS

In [ ]:
!uv run scripts/push_to_hub.py --ckpt checkpoints/cda_reg --repo $HF_USER/fairness-gpt2-qqp

Then in your Streamlit app → Settings → Secrets:

```toml
MODEL_REPO = "YOUR-HF-USERNAME/fairness-gpt2-qqp"
```

Live predictions turn on.

---

## 7. Optional — the other two models

`cda_reg` alone is enough to deploy. These only add the full Table 2 comparison.
Run **one at a time**, re-downloading results after each.

In [ ]:
!uv run fairness-train --mode baseline --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/baseline --epochs 5

In [ ]:
!uv run fairness-train --mode cda --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/cda --epochs 5

In [ ]:
!zip -qr reproduced.zip results/reproduced
from google.colab import files
files.download("reproduced.zip")

## What to compare against

At **5 epochs**, your report's Table 4 gives a direct comparison:

| Mode | Dev acc | Subgroup gap | Flip rate |
|---|---|---|---|
| CDA | 0.8835 | 0.0433 | 0.0392 |
| CDA + Reg. | 0.8856 | 0.0512 | 0.0296 |

Table 4 has no 5-epoch baseline — compare that against the 10-epoch row
(0.8955 / 0.0365 / 0.0456) and expect lower accuracy.

**Your flip rate will not match.** The report gives substitution *counts*
(60/20/22) and three examples, not the lists — `identity.py` reconstructs them.
Different lexicon, different counterfactuals. Accuracy should land close; flip
rate is the number that moves.